In [ ]:
#!unzip "/content/drive/MyDrive/Colab Notebooks/combined_dataset.zip" -d "/content/drive/MyDrive/Colab Notebooks/"

In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.2 MB/s eta 0:00:00


In [3]:
import torch
print("¿GPU disponible?:", torch.cuda.is_available())
print("Nombre de la GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Ninguna")

¿GPU disponible?: False
Nombre de la GPU: Ninguna


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from ultralytics import YOLO

def main():
    yaml_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/combined_dataset/", "data.yaml")

    if not os.path.exists(yaml_path):
        raise FileNotFoundError(
            f"Configuration file not found at: {os.path.abspath(yaml_path)}. "
            f"Please verify your repository directory structure."
        )

    print("[INFO] Initializing YOLOv8 Nano architecture...")
    model = YOLO("yolov8n.pt")

    print("[INFO] Starting training pipeline for the baseline approach (Dataset 2)...")

    results = model.train(
        data=yaml_path,
        epochs=40,
        imgsz=640,
        batch=16,
        device="gpu",
        workers=2,
        name="/content/drive/MyDrive/Colab Notebooks/Entrenamientos_YOLO",
        save=True,
        deterministic=True
    )

    print("\n" + "=" * 60)
    print("[INFO] Training pipeline successfully completed.")
    print(f"[INFO] Optimal weights saved to: runs/detect/yolov8_facial_emotion/weights/best.pt")
    print("=" * 60)

if __name__ == "__main__":
    main()

!zip -r resultados.zip /content/runs/

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Initializing YOLOv8 Nano architecture...
[INFO] Starting training pipeline for the baseline approach (Dataset 2)...
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:gpu (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=gpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None,

In [ ]:
import os
from ultralytics import YOLO

model_path = "/content/drive/MyDrive/Colab Notebooks/Entrenamientos_YOLO/weights/best.pt"
yaml_path = "/content/drive/MyDrive/Colab Notebooks/combined_dataset/data.yaml"
project_drive = "/content/drive/MyDrive/Colab Notebooks/Entrenamientos_YOLO"

print("[INFO] Cargando arquitectura de YOLOv8...")
model = YOLO(model_path)

print("[INFO] Iniciando validación oficial sobre el conjunto de TEST...")

metrics = model.val(
    data=yaml_path,
    split="test",
    device="cpu",
    project=project_drive,     
    name="resultados_test",     
    plots=True
)

print("\n" + "="*60)
print("[INFO] ¡Validación completada con éxito!")
print(f"[INFO] Gráficos y matrices guardados en: {project_drive}/resultados_test/")
print("="*60)

[INFO] Cargando arquitectura de YOLOv8...
[INFO] Iniciando validación oficial sobre el conjunto de TEST...
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.1 ms, read: 0.1±0.1 MB/s, size: 40.5 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/combined_dataset/test/labels.cache... 705 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 706/706 95.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 4.3s/it 3:15
                   all        706        738      0.851      0.872      0.909      0.635
                 angry        109        109      0.876      0.872      0.941      0.785
                 happy        104        127      0.816      0.835      0.889      0.354
                   sad        107        107      0.744      0.748        0.8      0.3

In [ ]:
import os
from ultralytics import YOLO

yaml_path = "/content/drive/MyDrive/Colab Notebooks/combined_dataset/data.yaml"
project_drive = "/content/drive/MyDrive/Colab Notebooks/Entrenamientos_v26"

print("[INFO] Starting the 3-Model comparative training pipeline...")

# =====================================================================
# EXPERIMENT 1: YOLO26 Nano at 640x640 (Architecture Comparison)
# =====================================================================
print("\n[RUN 1/3] Training YOLO26n at 640x640...")
model_26n = YOLO("yolo26n.pt")
model_26n.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    device="cuda:0",
    project=project_drive,
    name="YOLO26n_640_v_v8",
    deterministic=True,
    plots=True
)

# =====================================================================
# EXPERIMENT 2: YOLO26 Small at 640x640 (Model Capacity Scaling)
# =====================================================================
print("\n[RUN 2/3] Training YOLO26s at 640x640...")
model_26s = YOLO("yolo26s.pt")
model_26s.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    device="cuda:0",
    project=project_drive,
    name="YOLO26s_640_Scale",
    deterministic=True,
    plots=True
)

# =====================================================================
# EXPERIMENT 3: YOLO26 Nano at 1280x1280 (Spatial Resolution Scaling)
# =====================================================================
print("\n[RUN 3/3] Training YOLO26n at 1280x1280...")

model_26n_1280 = YOLO("yolo26n.pt")
model_26n_1280.train(
    data=yaml_path,
    epochs=40,
    imgsz=1280,
    batch=16,
    device="cuda:0",
    project=project_drive,
    name="YOLO26n_1280_HighRes",
    deterministic=True,
    plots=True
)

print("\n" + "="*60)
print("[INFO] All three experiments completed successfully!")
print("="*60)